# Gemma-3 Financial Fine-Tuning (QLoRA)

This notebook focuses on fine-tuning the **Gemma-3-1b-it** model using QLoRA on JAX for financial document analysis. We use the IMF World Economic Outlook report as our training material.

In [3]:
import os
import sys

# 1. Find and set NVIDIA Library Paths
conda_prefix = os.environ.get("CONDA_PREFIX")
if conda_prefix:
    site_packages = os.path.join(conda_prefix, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")
    nvidia_libs = []
    if os.path.exists(site_packages):
        for root, dirs, files in os.walk(os.path.join(site_packages, "nvidia")):
            if "lib" in dirs: nvidia_libs.append(os.path.join(root, "lib"))
    current_ld_path = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_libs + [current_ld_path]).strip(":")

# 2. GPU Memory Management (T4 Optimized)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"  # Limit JAX to 85% VRAM
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform" # Avoid aggressive pre-allocation
os.environ["JAX_LOG_COMPILES"] = "1" # Monitor compilation progress
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import jax
print(f"✅ JAX Device: {jax.devices()}")

✅ JAX Device: [CpuDevice(id=0)]


In [4]:
import jax
import numpy as np
import tunix
import qwix
import importlib.metadata

print(f"📦 Tunix: {importlib.metadata.version('google-tunix')}")
print(f"📦 Qwix: {importlib.metadata.version('qwix')}")
print(f"📦 NumPy: {np.__version__}")

📦 Tunix: 0.1.6
📦 Qwix: 0.1.5
📦 NumPy: 2.1.3


## 1. Authentication & Model Setup

In [5]:
import os
from huggingface_hub import login
from getpass import getpass

# Enter your token from https://huggingface.co/settings/tokens
hf_token = getpass("Enter Hugging Face Token: ")
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token)
print("✅ Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Hugging Face login successful.


In [6]:
import os
import optax
from flax import nnx
from huggingface_hub import snapshot_download
from tunix.models.gemma3 import model as gemma3_model_lib
from tunix.models.gemma3 import params_safetensors as params_safetensors_lib
from tunix.sft import peft_trainer

# 1. Download Model Weights
model_id = "google/gemma-3-1b-it"
local_model_path = snapshot_download(repo_id=model_id, ignore_patterns=["*.pth"])
model_config = gemma3_model_lib.ModelConfig.gemma3_1b_it()

# 2. Load Base Model
base_model = params_safetensors_lib.create_model_from_safe_tensors(local_model_path, model_config)

# 3. Configure QLoRA (Rank 32)
lora_provider = qwix.LoraProvider(
    module_path=".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj",
    rank=16, alpha=32.0, weight_qtype="nf4", tile_size=128,
)

rngs = nnx.Rngs(0)
model_input = base_model.get_model_input()
model = qwix.apply_lora_to_model(base_model, lora_provider, rngs=rngs, **model_input)

# 4. Initialize Trainer
optimizer = optax.adamw(learning_rate=5e-4)
checkpoint_path = os.path.abspath("./gemma3_lora_results")

trainer = peft_trainer.PeftTrainer(
    model=model,
    optimizer=optimizer,
    training_config=peft_trainer.TrainingConfig(
        eval_every_n_steps=50,
        max_steps=150,
        checkpoint_root_directory=checkpoint_path,
        data_sharding_axis=()
    )
)

print("✅ Model and LoRA architecture initialized.")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Fetching 10 files: 100%|██████████| 10/10 [00:06<00:00,  1.56it/s]


✅ Model and LoRA architecture initialized.


In [7]:
from tunix.generate import tokenizer_adapter as tokenizer_lib

GEMMA_TOKENIZER_PATH = os.path.join(local_model_path, "tokenizer.model")
tokenizer = tokenizer_lib.Tokenizer(tokenizer_path=GEMMA_TOKENIZER_PATH)
print("✅ Tokenizer loaded.")

✅ Tokenizer loaded.


## 2. Financial Data Engineering

In [8]:
import fitz  # PyMuPDF
import requests
import json
from tqdm import tqdm

# 1. Download IMF Report
pdf_url = "https://www.imf.org/-/media/Files/Publications/WEO/2024/October/English/text.pdf"
pdf_path = "imf_weo_2024.pdf"

if not os.path.exists(pdf_path):
    print("📥 Downloading IMF World Economic Outlook...")
    response = requests.get(pdf_url, stream=True)
    with open(pdf_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

# 2. Extract Text & Create JSONL
print("📄 Converting PDF to training format...")
dataset = []
doc = fitz.open(pdf_path)
for i in range(0, len(doc), 2):
    content = "".join([p.get_text() for p in doc[i : i+2]])
    dataset.append({
        'instruction': "Analyze this IMF report segment and summarize key economic projections or policy advice.",
        'input': f"[Document: Pages {i+1}-{i+2}]\n\n{content[:3000]}",
        'output': f"This segment discusses {content[:100].strip()}... The IMF highlights..."
    })

train_file = "imf_train.jsonl"
with open(train_file, "w", encoding="utf-8") as f:
    for entry in dataset:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"✅ Generated {len(dataset)} training entries in {train_file}")

📥 Downloading IMF World Economic Outlook...
📄 Converting PDF to training format...
✅ Generated 87 training entries in imf_train.jsonl


## 2.5 Baseline Inference (Before Training)

Let"s see how the base **Gemma-3** model performs on financial questions before we inject the IMF report knowledge.

In [9]:
from tunix.generate import sampler as sampler_lib

# Initialize sampler with the base (untrained) model
base_sampler = sampler_lib.Sampler(
    transformer=model,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=256, num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads, head_dim=model_config.head_dim,
    )
)

baseline_prompts = [
    "Summarize the global growth risks mentioned in the October 2024 IMF report.",
    "What are the IMF's core policy recommendations for emerging markets in late 2024?"
]

formatted_baseline = [f"<start_of_turn>user\n{p}<end_of_turn>\n<start_of_turn>model\n" for p in baseline_prompts]

print("📉 Baseline Results (Before Training):")
baseline_results = base_sampler(input_strings=formatted_baseline, max_generation_steps=150, eos_tokens=[tokenizer.eos_id()])

for q, a in zip(baseline_prompts, baseline_results.text):
    print(f"\n--- Question: {q}\n--- Answer: {a.strip()}")

📉 Baseline Results (Before Training):



--- Question: Summarize the global growth risks mentioned in the October 2024 IMF report.
--- Answer: Okay, here’s a summary of the global growth risks highlighted in the October 2024 IMF report, focusing on key takeaways and trends:

**Overall Picture: A More Uncertain Global Outlook**

The IMF’s October 2024 report paints a significantly more cautious picture of global economic growth compared to earlier in the year. They’ve shifted from a scenario of “moderate” growth to one of “slow and uneven” growth, with significant uncertainty looming.

**Key Growth Risks & Themes:**

1. **Regional Disparities:**
   * **China’s Slowdown:** China’s growth is slowing significantly, and the IMF is projecting a more severe slowdown in the next few years. This is

--- Question: What are the IMF's core policy recommendations for emerging markets in late 2024?
--- Answer: Okay, let's break down the IMF's core policy recommendations for emerging markets in late 2024. It's a complex and evolving landsc

In [10]:
import gc
import jax

# Clean up Baseline Inference resources to free VRAM for training
if "base_sampler" in globals():
    del base_sampler

gc.collect()
# Optional: This clears JAX compiled function cache to save more memory
# jax.clear_backends_cache() 
print("🧹 VRAM cleaned up for training session.")

🧹 VRAM cleaned up for training session.


## 3. Training Execution

In [11]:
import numpy as np
from tunix.sft import utils

# --- JAX-Native Data Loader (No TensorFlow) ---
class FinancialDataLoader:
    def __init__(self, file_path, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.batches = []
        pad_id = self.tokenizer.pad_id()
            
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                entry = json.loads(line)
                full_text = f"<start_of_turn>user\n{entry['instruction']}\n{entry['input']}<end_of_turn>\n"
                full_text += f"<start_of_turn>model\n{entry['output']}<end_of_turn>"
                tokens = tokenizer.encode(full_text)
                
                arr = np.full((1, self.max_length), pad_id, dtype=np.int32)
                ln = min(len(tokens), self.max_length)
                arr[0, :ln] = tokens[:ln]
                
                self.batches.append(peft_trainer.TrainingInput(
                    input_tokens=arr, 
                    input_mask=(arr != pad_id)
                ))

    def __len__(self): return len(self.batches)
    def __iter__(self): return iter(self.batches)

def financial_input_fn(x: peft_trainer.TrainingInput):
    return {
        "input_tokens": x.input_tokens,
        "input_mask": x.input_mask,
        "positions": utils.build_positions_from_mask(x.input_mask),
        "attention_mask": utils.make_causal_attn_mask(x.input_mask),
    }

trainer = trainer.with_gen_model_input_fn(financial_input_fn)
financial_ds = FinancialDataLoader(train_file, tokenizer, max_length=512)
print(f"🚀 Loaded {len(financial_ds)} financial training batches.")

🚀 Loaded 87 financial training batches.


In [12]:
print("🔥 Starting Financial Knowledge Injection...")
try:
    trainer.train(financial_ds, [])
    print("\n✅ Fine-tuning completed successfully!")
except Exception as e:
    print(f"\n❌ Training Error: {e}")

🔥 Starting Financial Knowledge Injection...


Training:   1%|▏         | 2/150 [01:56<2:08:57, 52.28s/step, _train_loss=4.99, _train_perplexity=147, _train_steps_per_sec=0.023]

: 

## 4. Financial Expert Inference

In [ ]:
from tunix.generate import sampler as sampler_lib

sampler = sampler_lib.Sampler(
    transformer=model,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=256, num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads, head_dim=model_config.head_dim,
    )
)

test_prompts = [
    "Summarize the global growth risks mentioned in the October 2024 IMF report.",
    "What are the IMF's core policy recommendations for emerging markets in late 2024?"
]

formatted_prompts = [f"<start_of_turn>user\n{p}<end_of_turn>\n<start_of_turn>model\n" for p in test_prompts]

print("🏆 Financial Expert Mode Results:")
results = sampler(input_strings=formatted_prompts, max_generation_steps=150, eos_tokens=[tokenizer.eos_id()])

for q, a in zip(test_prompts, results.text):
    print(f"\n--- Question: {q}\n--- Answer: {a.strip()}")